# ForgeEdge — Event Discovery

Questo notebook illustra l'utilizzo del modulo **EventDiscovery**, il primo passo della pipeline FORGE.

Il modulo prende in input una tabella di KPI (indicatori tecnici su OHLCV) e restituisce una lista di **Event Candidates**: condizioni booleane sulle serie temporali che si attivano in modo statisticamente consistente nel tempo.

## Pipeline interna

```
Step 0 — TypeClassifier    → classifica le colonne (CONTINUOUS / BINARY / CATEGORICAL)
Step 1 — FeatureGenerator  → genera feature derivate (ratio, spread_pct, diffnorm, bb_pct_b, ...)
Step 2 — TransformLayer    → applica trasformate (identity, pctrank, zscore, delta)
Step 3 — EventGenerator    → converte le serie in eventi booleani (threshold + crossing)
Step 4 — ConsistencyGate   → filtra gli eventi per volume, copertura, concentrazione, frequenza
Step 5 — ANDComposer       → combina coppie/triple di eventi con AND logico + ri-applica il gate
```


## 0. Setup

In [ ]:
import sys
sys.path.insert(0, "../src")   # per esecuzione da notebooks/

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

from forgedge import EventDiscovery, DiscoveryConfig
from forgedge.event_discovery.models import GateParams

## 1. Caricamento e preprocessing

Il dataset contiene candele orarie per due simboli (ADAUSDC, DOGEUSDC).  
Filtriamo su un singolo simbolo e lasciamo che EventDiscovery gestisca la conversione del timestamp.

> **Nota**: il modulo rileva automaticamente l'unità del timestamp numerico (s / ms / us / ns)  
> a partire dal valore mediano della colonna. Non è necessario convertire manualmente.

In [ ]:
# Sostituisci con il percorso al tuo file
DATA_PATH = "../data/test1h.xlsx"
SYMBOL = "ADAUSDC"

df = pd.read_excel(DATA_PATH)
df = df[df["symbol"] == SYMBOL].copy()
df = df.sort_values("open_time")

print(f"Simbolo : {SYMBOL}")
print(f"Righe   : {len(df):,}")
print(f"Colonne : {list(df.columns)}")
print(f"\nPrime righe:")
df.head(3)

## 2. Configurazione del pipeline

`DiscoveryConfig` raccoglie tutti i parametri:

| Parametro | Default | Descrizione |
|---|---|---|
| `timestamp_col` | `"open_dt"` | Nome della colonna timestamp (int, datetime o string) |
| `gate_params.min_act` | 50 | Attivazioni totali minime |
| `gate_params.min_months` | 8 | Mesi distinti minimi con almeno 1 attivazione |
| `gate_params.max_conc` | 0.40 | Max quota attivazioni in un singolo mese |
| `gate_params.min_tpm` | 2.0 | Attivazioni medie per mese minime |
| `max_and_components` | 2 | Cardinalità massima delle composizioni AND (2 o 3) |
| `scale_free_overrides` | None | Override manuali per il rilevamento scale-free |

In [ ]:
config = DiscoveryConfig(
    timestamp_col="open_time",
    gate_params=GateParams(
        min_act=50,
        min_months=8,
        max_conc=0.40,
        min_tpm=2.0,
    ),
    max_and_components=2,
)

## 3. Esecuzione del pipeline

In [ ]:
ed = EventDiscovery(df, config)
candidates = ed.run()

print(f"Candidati totali : {len(candidates)}")
print(f"  di cui singoli : {sum(1 for c in candidates if len(c.components) == 1)}")
print(f"  di cui AND     : {sum(1 for c in candidates if len(c.components) > 1)}")

## 4. Summary DataFrame

`ed.summary()` restituisce un DataFrame piatto con le metriche principali di ogni candidato.

In [ ]:
summary_df = ed.summary()
summary_df.head(5)

### Costruzione manuale del summary con metriche aggiuntive

Utile per aggiungere colonne custom (es. `arity`, `transform`, `source`) non presenti nel summary standard.

In [ ]:
records = []
for c in candidates:
    g = c.consistency_gate    # GateResult
    s = c.activation_stats    # ActivationStats (include zero_months)
    comp = c.components[0]    # primo EventComponent
    records.append({
        "expression":  c.expression,
        "arity":       len(c.components),
        "n_act":       g.n_activations,
        "n_months":    g.n_active_months,
        "zero_months": s.zero_months,
        "max_conc":    round(g.max_monthly_share, 4),
        "tpm":         round(g.mean_tpm, 2),
        "source":      comp.source_feature,
        "transform":   comp.transform,
        "event_type":  comp.event_type,
        "_candidate":  c,
    })

summary = pd.DataFrame(records)
singles = summary[summary["arity"] == 1].sort_values("max_conc")
ands    = summary[summary["arity"] > 1].sort_values(["max_conc", "n_act"], ascending=[True, False])

print(f"Singles: {len(singles)} | ANDs: {len(ands)}")

## 5. Migliori eventi singoli

Ordinati per **concentrazione mensile minima** (`max_conc`): valori bassi indicano  
attivazioni ben distribuite nel tempo — segnali più robusti e non legati a un singolo periodo.

In [ ]:
cols = ["expression", "n_act", "n_months", "zero_months", "max_conc", "tpm", "transform"]
singles.head(15)[cols]

### Distribuzione per tipo di trasformata

In [ ]:
singles.groupby("transform")["n_act"].agg(["count", "mean", "median"]).round(1)

## 6. Migliori composizioni AND

Coppie di eventi singoli combinate con AND logico che superano il ConsistencyGate.  
La composizione AND è più selettiva (meno attivazioni) ma cattura condizioni di mercato più specifiche.

In [ ]:
cols_and = ["expression", "n_act", "n_months", "zero_months", "max_conc", "tpm"]
ands.head(15)[cols_and]

## 7. Ispezione di un candidato

Ogni `EventCandidate` espone:
- `expression` — formula leggibile della condizione
- `components` — lista di `EventComponent` (uno per ogni sotto-condizione nell'AND)
- `consistency_gate` — `GateResult` con le metriche del filtro
- `activation_stats` — `ActivationStats` con `zero_months`
- `event_series` — serie booleana (0/1/NaN) con DatetimeIndex, resample-ready

In [ ]:
best = ands.iloc[0]["_candidate"]

print(f"Expression  : {best.expression}")
print(f"\nGate metrics:")
g = best.consistency_gate
print(f"  n_activations    : {g.n_activations}")
print(f"  n_active_months  : {g.n_active_months}")
print(f"  zero_months      : {best.activation_stats.zero_months}")
print(f"  max_monthly_share: {g.max_monthly_share:.4f}")
print(f"  mean_tpm         : {g.mean_tpm:.2f}")
print(f"\nComponents ({len(best.components)}):")
for i, comp in enumerate(best.components):
    print(f"  [{i}] transform={comp.transform:<20} expression={comp.expression}")

## 8. Breakdown mensile

`event_series` ha il DatetimeIndex già impostato → `.resample()` funziona direttamente.

In [ ]:
monthly = best.event_series.resample("ME").sum()
monthly.index = monthly.index.strftime("%Y-%m")

monthly.plot(
    kind="bar",
    figsize=(14, 4),
    color="steelblue",
    width=0.8,
    title=f"Attivazioni mensili — {best.expression}",
    ylabel="Attivazioni",
    xlabel="Mese",
)

In [ ]:
# Tabella mensile per i candidati con più bassa concentrazione
print("Mesi non-zero:")
print(monthly[monthly > 0].to_string())

## 9. Classificazioni delle colonne

`ed.get_classifications()` restituisce il dizionario prodotto da `TypeClassifier` (Step 0).  
Utile per verificare quali colonne sono state riconosciute come scale-free e quali escluse.

In [ ]:
clf_dict = ed.get_classifications()

clf_rows = []
for col, clf in sorted(clf_dict.items()):
    clf_rows.append({
        "column":      col,
        "type":        clf.col_type.name,
        "n_distinct":  clf.n_distinct,
        "scale_free":  clf.effective_scale_free if clf.col_type.name == "CONTINUOUS" else None,
    })

pd.DataFrame(clf_rows)

## 10. Feature table di debug

`ed.df` è il DataFrame completo post-pipeline con:
- **DatetimeIndex** impostato dal modulo
- Le colonne originali (OHLCV, indicatori)
- Tutte le **feature derivate** generate da `FeatureGenerator` (ratio, diffnorm, spread_pct, bb_pct_b…)

Utile per ispezionare le feature calcolate o tracciare grafici delle serie sottostanti.

In [ ]:
print(f"Shape : {ed.df.shape}")
print(f"Index : {type(ed.df.index).__name__}  {ed.df.index[0]} → {ed.df.index[-1]}")

derived = [c for c in ed.df.columns if any(
    c.startswith(p) for p in ("ratio_", "spread_", "diffnorm_", "bb_", "rng_")
)]
print(f"\nFeature derivate ({len(derived)}):")
for col in derived:
    print(f"  {col}")

In [ ]:
# Esempio: visualizza una feature derivata usata nei top candidati
import matplotlib.pyplot as plt

col_to_plot = "diffnorm_close_rsi14_rsi25"
if col_to_plot in ed.df.columns:
    fig, ax = plt.subplots(figsize=(14, 3))
    ed.df[col_to_plot].plot(ax=ax, lw=0.7, color="darkorange")
    ax.set_title(f"Feature derivata: {col_to_plot}")
    ax.set_ylabel("Valore")
    ax.axhline(0, color="gray", lw=0.5, linestyle="--")
    plt.tight_layout()

## 11. Confronto tra simboli (opzionale)

Confronta il numero e la qualità dei candidati tra ADAUSDC e DOGEUSDC.

In [ ]:
results = {}
df_all = pd.read_excel(DATA_PATH)

for sym in ["ADAUSDC", "DOGEUSDC"]:
    df_sym = df_all[df_all["symbol"] == sym].copy().sort_values("open_time")
    ed_sym = EventDiscovery(df_sym, config)
    cands  = ed_sym.run()
    results[sym] = {
        "total":   len(cands),
        "singles": sum(1 for c in cands if len(c.components) == 1),
        "ands":    sum(1 for c in cands if len(c.components) > 1),
        "best_max_conc": min(c.consistency_gate.max_monthly_share for c in cands) if cands else float("nan"),
    }

pd.DataFrame(results).T